# Aquaview MCP with ChatGPT (OpenAI Responses API)

This notebook shows how to give ChatGPT direct access to the Aquaview MCP catalog using the OpenAI Responses API's native hosted-MCP tool — no proxy, no local server.

We'll ask ChatGPT the same question as in the Claude and Gemini notebooks:

> *Find Argo float profiles within 200 km of Hawaii from March 2026 where the float reached at least 1500 dbar pressure. Summarize the temperature range observed and give me a few download links.*

ChatGPT will autonomously call Aquaview's `search_datasets`, `get_item`, and possibly `aggregate` tools to answer.

**Sources used**: `GADR` (Global Argo Data Repository, NOAA NCEI)

## Install

You only need the `openai` SDK.

In [ ]:
%pip install -U openai

## Setup

Set your OpenAI API key in the `OPENAI_API_KEY` environment variable.

In [ ]:
import os
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your environment"

client = OpenAI()

AQUAVIEW_TOOL = {
    "type": "mcp",
    "server_label": "aquaview",
    "server_url": "https://mcp.aquaview.org/mcp",
    "require_approval": "never",
}

MODEL = "gpt-5"

## Ask the question

We pass the Aquaview MCP tool in `tools=[...]` and the model can call it directly. ChatGPT will issue tool calls, the API will execute them against `mcp.aquaview.org/mcp`, and the model will integrate the results into its final answer.

In [ ]:
PROMPT = (
    "Find Argo float profiles within 200 km of Hawaii from March 2026 "
    "where the float reached at least 1500 dbar pressure. Summarize the "
    "temperature range observed and give me a few download links."
)

response = client.responses.create(
    model=MODEL,
    input=PROMPT,
    tools=[AQUAVIEW_TOOL],
)

print(f"status: {response.status}")
print(f"input  tokens: {response.usage.input_tokens}")
print(f"output tokens: {response.usage.output_tokens}")

## Inspect what ChatGPT did

The Responses API returns a sequence of `output` items including:

- `mcp_list_tools` — when the model first inspected the Aquaview tool schema
- `mcp_call` — each tool call with input arguments and returned content
- `message` — the model's final response text

In [ ]:
from textwrap import shorten

for i, item in enumerate(response.output):
    item_type = getattr(item, "type", type(item).__name__)
    if item_type == "mcp_list_tools":
        print(f"[{i}] mcp_list_tools (server: {item.server_label})")
    elif item_type == "mcp_call":
        args = getattr(item, "arguments", {}) or {}
        out = getattr(item, "output", "") or ""
        preview = shorten(str(out).replace("\n", " "), width=240, placeholder="…")
        print(f"[{i}] mcp_call → aquaview.{item.name}({args})")
        print(f"    {preview}")
    elif item_type == "message":
        text = "\n".join(
            getattr(c, "text", "")
            for c in (item.content or [])
            if getattr(c, "type", "") == "output_text"
        )
        print(f"[{i}] message\n{text}\n")
    else:
        print(f"[{i}] {item_type}")

Typical trace for this prompt:

1. `mcp_list_tools` — model fetches Aquaview's tool schemas
2. `mcp_call → aquaview.search_datasets(collections='GADR', bbox='-161,18,-154,23', datetime='2026-03-01.../2026-03-31...', filter={...Pressure.max >= 1500...})`
3. `mcp_call → aquaview.get_item(collection='GADR', item_id='argo_jma_7900869')` to fetch download URLs
4. `message` — ChatGPT's narrative summary citing the floats and their NCEI / Ifremer GDAC URLs

## Render ChatGPT's final answer

In [ ]:
from IPython.display import Markdown, display

display(Markdown(response.output_text))

## Multi-turn — follow up

Pass the prior `response.id` as `previous_response_id` to keep the conversation context (and any prior MCP tool results) on the server side.

In [ ]:
followup = client.responses.create(
    model=MODEL,
    previous_response_id=response.id,
    input=(
        "For the deepest-diving float you found, give me its full lifetime "
        "trajectory bounding box and the direct NetCDF download URL."
    ),
    tools=[AQUAVIEW_TOOL],
)

display(Markdown(followup.output_text))

## Streaming

For interactive UIs, stream the response so you can render the model's reasoning + tool calls + final text as they arrive.

In [ ]:
with client.responses.stream(
    model=MODEL,
    input=(
        "How many Argo profile observations are in the Aquaview catalog "
        "from 2025? Use an aggregation, don't enumerate."
    ),
    tools=[AQUAVIEW_TOOL],
) as stream:
    for event in stream:
        et = getattr(event, "type", "")
        if et == "response.output_text.delta":
            print(event.delta, end="", flush=True)
        elif et == "response.mcp_call.in_progress":
            print(f"\n[tool] aquaview.{event.item.name}", flush=True)
    print()

## Where to go next

- **Same notebook in Claude / Gemini**: see [`01-claude-mcp-agent.ipynb`](01-claude-mcp-agent.ipynb) and [`03-gemini-mcp-agent.ipynb`](03-gemini-mcp-agent.ipynb) — same prompt, three different agent runtimes.
- **Other examples**: see [`../examples/`](../examples/) for 20+ more research questions, with prompts and transcripts.
- **Tool reference**: full parameter docs in [`../docs/tools-reference.md`](../docs/tools-reference.md).
- **OpenAI Agents SDK (alternative)**: if you want multi-step orchestration with handoffs, the OpenAI Agents SDK supports MCP via `MCPServerStreamableHttp` — see the [INSTALL.md](../INSTALL.md) snippet.